# PDEformer Finetuning: `MHD_64`

This notebook is a dataset-specific Well fine-tuning launcher in the style of `PDEformer_finetune_demo.ipynb`. It uses the equation registry in `src.data.well_equations` so PDEformer receives the `MHD_64` PDE DAG instead of the generic Well placeholder.

**Spatial dimensions:** 3

**Default selected fields:** `density, magnetic_field_x, magnetic_field_y, magnetic_field_z, velocity_x, velocity_y, velocity_z`


## Equation

$\rho_t+\nabla\cdot(\rho v)=0$\n$(\rho v)_t+\nabla\cdot(\rho vv-BB)+\nabla p=0$\n$B_t-\nabla\times(v\times B)=0$

**Registry note:** Dataset-specific equation DAG registered in `src.data.well_equations`.


## PDEformer-2 Support Note

This Well dataset is not 2D. The equation is registered for metadata/notebook completeness, but the current PDEformer-2 Well fine-tuning adapter intentionally raises for non-2D grids.


In [ ]:
from dataclasses import dataclass
from pathlib import Path

from src.data.well_equations import get_well_equation_spec

WELL_DATASET = "MHD_64"
WELL_BASE_PATH = Path("/path/to/the_well")
TRAIN_SPLIT = "train"
TEST_SPLIT = "valid"
CHECKPOINT = Path("model-L.pt")
FIELD_INDICES = "0,1,2,3,4,5,6"
MAX_FIELDS = 7

spec = get_well_equation_spec(WELL_DATASET)
print(spec.pde_latex)


## Slurm Finetuning

Set `WELL_BASE_PATH` to the root containing the downloaded Well datasets. The Slurm wrapper writes a concrete YAML config into `slurm_logs/` and then runs `train.py`.


In [ ]:
slurm_command = f'''sbatch --export=ALL,\
WELL_BASE_PATH={WELL_BASE_PATH},\
WELL_DATASET={WELL_DATASET},\
TRAIN_SPLIT={TRAIN_SPLIT},\
TEST_SPLIT={TEST_SPLIT},\
CHECKPOINT={CHECKPOINT},\
FIELD_INDICES={FIELD_INDICES},\
MAX_FIELDS={MAX_FIELDS} \
scripts/submit_the_well_finetune_slurm.sh'''
print(slurm_command)


## Local Smoke Test

Use a tiny sample count and `WANDB_MODE=offline` for a pipeline check before submitting a larger job.


In [ ]:
local_command = f'''WELL_BASE_PATH={WELL_BASE_PATH} \
WELL_DATASET={WELL_DATASET} \
TRAIN_SAMPLES=1 TEST_SAMPLES=1 EPOCHS=0 \
FIELD_INDICES={FIELD_INDICES} MAX_FIELDS={MAX_FIELDS} \
WANDB_MODE=offline DEVICE_TARGET=CPU DEVICE_ID=0 \
bash scripts/submit_the_well_finetune_slurm.sh'''
print(local_command)


## Equation-Aware Evaluation

The evaluator also defaults to `--pde-preset well_equation`, so the same registered equation DAG is used for raw or fine-tuned checkpoint checks.


In [ ]:
eval_command = f'''python scripts/evaluate_the_well.py \
  --config configs/inference/model-L.yaml \
  --checkpoint {CHECKPOINT} \
  --well-base-path {WELL_BASE_PATH} \
  --well-dataset {WELL_DATASET} \
  --split test \
  --num-samples 4 \
  --field-indices {FIELD_INDICES} \
  --pde-preset well_equation \
  --output exp/the_well/{WELL_DATASET}_test_equation_eval.json'''
print(eval_command)
